In [11]:
%%writefile doctor.py
"""AI Dataset Doctor - core engine: diagnose, recommend, treat, train, explain."""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    SMOTE = None
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None


# ----------------------------------------------------------------- 1. DIAGNOSE
def diagnose(df: pd.DataFrame, target: str, corr_thr=0.85, z_thr=3.0) -> dict:
    rep = {"rows": len(df), "cols": df.shape[1]}
    X = df.drop(columns=[target])

    # Missing values (blank strings count as missing)
    blank = df.replace(r"^\s*$", np.nan, regex=True)
    miss = blank.isna().sum()
    rep["missing"] = (miss[miss > 0] / len(df) * 100).round(2).to_dict()

    # Duplicates
    rep["duplicates"] = int(df.duplicated().sum())

    # Wrong data types: numbers stored as text
    wrong = {}
    for c in X.select_dtypes(include="object"):
        s = blank[c].dropna()
        if len(s) and pd.to_numeric(s, errors="coerce").notna().mean() >= 0.9:
            wrong[c] = "object -> numeric"
    rep["wrong_dtypes"] = wrong

    num = X.drop(columns=list(wrong), errors="ignore").select_dtypes(include="number")
    num_all = pd.concat([num, X[list(wrong)].apply(pd.to_numeric, errors="coerce")], axis=1) if wrong else num

    # Outliers: IQR and Z-score
    out = {}
    for c in num_all:
        s = num_all[c].dropna()
        if s.nunique() <= 2 or s.empty:
            continue
        q1, q3 = s.quantile([.25, .75])
        iqr = q3 - q1
        n_iqr = int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())
        n_z = int((np.abs((s - s.mean()) / (s.std() or 1)) > z_thr).sum())
        if n_iqr or n_z:
            out[c] = {"iqr": n_iqr, "zscore": n_z}
    rep["outliers"] = out

    # Class imbalance
    vc = df[target].value_counts(normalize=True)
    rep["class_dist"] = (vc * 100).round(2).to_dict()
    rep["imbalance_ratio"] = round(vc.max() / vc.min(), 2) if len(vc) > 1 else 1.0

    # Highly correlated features
    pairs = []
    if num_all.shape[1] > 1:
        corr = num_all.corr().abs()
        for i, a in enumerate(corr.columns):
            for b in corr.columns[i + 1:]:
                if corr.loc[a, b] >= corr_thr:
                    pairs.append((a, b, round(float(corr.loc[a, b]), 3)))
    rep["high_corr"] = pairs

    # Irrelevant features: ID-like, constant, near-constant
    irr = {}
    for c in X:
        n = X[c].nunique(dropna=True)
        if n <= 1:
            irr[c] = "constant"
        elif n / len(X) > 0.95 and (X[c].dtype == "object" or "id" in c.lower()):
            irr[c] = "ID-like (unique per row)"
        elif X[c].dtype == "object" and X[c].value_counts(normalize=True).iloc[0] > 0.99:
            irr[c] = "near-constant"
    rep["irrelevant"] = irr

    # PCA: components needed for 95% variance
    if num_all.shape[1] > 1:
        Z = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(num_all))
        ev = PCA().fit(Z).explained_variance_ratio_.cumsum()
        rep["pca_components_95"] = int(np.argmax(ev >= 0.95) + 1)
        rep["pca_curve"] = ev.round(4).tolist()

    rep["health_score"] = _score(rep)
    return rep


def _score(r) -> int:
    s = 100.0
    s -= min(20, sum(r["missing"].values()) / max(len(r["missing"]), 1) * 2 + len(r["missing"]) * 2)
    s -= min(10, r["duplicates"] / r["rows"] * 100 * 2)
    s -= min(10, len(r["wrong_dtypes"]) * 5)
    s -= min(15, len(r["outliers"]) * 3)
    s -= min(20, (r["imbalance_ratio"] - 1) * 4)
    s -= min(10, len(r["high_corr"]) * 4)
    s -= min(10, len(r["irrelevant"]) * 5)
    return int(max(0, round(s)))


# -------------------------------------------------------------- 2. RECOMMEND
def recommend(rep: dict) -> list[str]:
    tips = []
    if rep["wrong_dtypes"]:
        tips.append(f"Convert to numeric: {', '.join(rep['wrong_dtypes'])} (blank strings become NaN).")
    if rep["missing"]:
        tips.append("Impute missing values: median for numeric, most-frequent for categorical "
                    "(drop columns with >40% missing).")
    if rep["duplicates"]:
        tips.append(f"Drop {rep['duplicates']} duplicate rows.")
    if rep["outliers"]:
        tips.append("Winsorize outliers with IQR clipping, or use tree models that are robust to them.")
    if rep["imbalance_ratio"] > 1.5:
        tips.append(f"Class imbalance {rep['imbalance_ratio']}:1 -> apply SMOTE on training data only "
                    "and judge models by F1 / ROC-AUC, not accuracy.")
    if rep["high_corr"]:
        tips.append("Drop one feature from each highly correlated pair: "
                    + ", ".join(f"{a}~{b}" for a, b, _ in rep["high_corr"]) + ".")
    if rep["irrelevant"]:
        tips.append(f"Remove irrelevant features: {', '.join(rep['irrelevant'])}.")
    if rep.get("pca_components_95") and rep["pca_components_95"] < len(rep.get("pca_curve", [])):
        tips.append(f"PCA: {rep['pca_components_95']} components explain 95% of numeric variance "
                    "(optional dimensionality reduction).")
    return tips or ["Dataset looks healthy - ready for modelling."]


def suggest_algorithms(rep: dict) -> list[str]:
    algos = ["Random Forest (robust to outliers, handles mixed features)",
             "XGBoost (strong on tabular data, handles imbalance via scale_pos_weight)"]
    if rep["rows"] < 5000:
        algos.append("Logistic Regression (good baseline for small datasets)")
    if rep["imbalance_ratio"] > 3:
        algos.append("Balanced Random Forest / cost-sensitive learning")
    return algos


# ------------------------------------------------------------------ 3. TREAT
def treat(df: pd.DataFrame, target: str, rep: dict, clip_outliers=True) -> pd.DataFrame:
    d = df.replace(r"^\s*$", np.nan, regex=True).drop_duplicates().copy()
    for c in rep["wrong_dtypes"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna(subset=[target])
    drop = set(rep["irrelevant"]) | {b for _, b, _ in rep["high_corr"]}
    drop |= {c for c in d.columns if c != target and d[c].isna().mean() > 0.4}
    d = d.drop(columns=[c for c in drop if c in d.columns and c != target])
    if clip_outliers:
        for c in rep["outliers"]:
            if c in d:
                q1, q3 = d[c].quantile([.25, .75])
                i = q3 - q1
                d[c] = d[c].clip(q1 - 1.5 * i, q3 + 1.5 * i)
    return d


# ------------------------------------------------------------ 4. TRAIN / EVAL
def _preprocessor(X):
    num = X.select_dtypes(include="number").columns.tolist()
    cat = [c for c in X.columns if c not in num]
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
    ])


def _models():
    m = {"Logistic Regression": LogisticRegression(max_iter=1000),
         "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)}
    if XGBClassifier:
        m["XGBoost"] = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                     eval_metric="logloss", random_state=42)
    return m


def train_evaluate(df: pd.DataFrame, target: str, use_smote=False, seed=42):
    """Returns (results DataFrame, dict with fitted models + test data for SHAP)."""
    y = LabelEncoder().fit_transform(df[target].astype(str))
    X = df.drop(columns=[target])
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

    pre = _preprocessor(Xtr)
    Xtr_t, Xte_t = pre.fit_transform(Xtr), pre.transform(Xte)
    if use_smote and SMOTE is not None:
        Xtr_t, ytr = SMOTE(random_state=seed).fit_resample(Xtr_t, ytr)

    rows, fitted = [], {}
    binary = len(np.unique(y)) == 2
    for name, model in _models().items():
        model.fit(Xtr_t, ytr)
        pred = model.predict(Xte_t)
        proba = model.predict_proba(Xte_t)
        rows.append({"Model": name,
                     "Accuracy": accuracy_score(yte, pred),
                     "F1": f1_score(yte, pred, average="binary" if binary else "weighted"),
                     "ROC-AUC": roc_auc_score(yte, proba[:, 1] if binary else proba,
                                              multi_class="ovr")})
        fitted[name] = model
    res = pd.DataFrame(rows).round(4).sort_values("F1", ascending=False).reset_index(drop=True)
    names = pre.get_feature_names_out()
    return res, {"models": fitted, "X_test": pd.DataFrame(Xte_t, columns=names)}


# ------------------------------------------------------------------ 5. SHAP
def shap_values(model, X_test: pd.DataFrame, n=300):
    import shap
    Xs = X_test.sample(min(n, len(X_test)), random_state=0)
    sv = shap.TreeExplainer(model).shap_values(Xs)
    if isinstance(sv, list):          # older shap: list per class
        sv = sv[1]
    elif getattr(sv, "ndim", 2) == 3:  # newer shap: (n, features, classes)
        sv = sv[:, :, 1]
    return sv, Xs

Overwriting doctor.py


In [12]:
!pip install streamlit

In [14]:
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import streamlit as st
import os

import doctor as dd

st.set_page_config(page_title="AI Dataset Doctor", page_icon="ᾝ1‍⚕️", layout="wide")
st.title("ᾝ1‍⚕️ AI Dataset Doctor")
st.caption("Check whether your data is healthy and ML-ready before training anything.")

f = st.sidebar.file_uploader("Upload a CSV (e.g. Kaggle Telco Customer Churn)", type="csv")

# Fallback mechanism so the notebook cell doesn't crash in standard execution mode
if f is None:
    sample_path = "/content/sample_data/california_housing_test.csv"
    if os.path.exists(sample_path):
        st.info(f"Running in notebook baseline mode. Loaded sample dataset: {sample_path}")
        df = pd.read_csv(sample_path)
    else:
        st.info("Upload a CSV to begin. Suggested: Kaggle → *blastchar/telco-customer-churn*.")
        st.stop()
else:
    df = pd.read_csv(f)

target = st.sidebar.selectbox("Target column", df.columns, index=len(df.columns) - 1)
use_smote = st.sidebar.checkbox("Apply SMOTE (train set only)", True)
clip = st.sidebar.checkbox("Clip outliers (IQR)", True)

rep = dd.diagnose(df, target)
t1, t2, t3, t4 = st.tabs(["ᾧA Health Report", "ὈA Prescription", "Ὄ8 Model Evaluation", "ὐD SHAP"])

with t1:
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Health score", f"{rep['health_score']}/100")
    c2.metric("Rows × Cols", f"{rep['rows']} × {rep['cols']}")
    c3.metric("Duplicates", rep["duplicates"])
    c4.metric("Imbalance ratio", f"{rep['imbalance_ratio']}:1")

    a, b = st.columns(2)
    with a:
        st.subheader("Missing values (%)")
        st.write(rep["missing"] or "None ✅")
        st.subheader("Wrong data types")
        st.write(rep["wrong_dtypes"] or "None ✅")
        st.subheader("Irrelevant features")
        st.write(rep["irrelevant"] or "None ✅")
    with b:
        st.subheader("Class distribution")
        cd = pd.Series(rep["class_dist"]).rename_axis("class").reset_index(name="%")
        st.plotly_chart(px.bar(cd, x="class", y="%", color="class"), use_container_width=True)

    st.subheader("Outliers (IQR / Z-score counts)")
    st.dataframe(pd.DataFrame(rep["outliers"]).T if rep["outliers"] else pd.DataFrame())
    st.subheader("Highly correlated pairs")
    st.write(rep["high_corr"] or "None ✅")
    if "pca_curve" in rep:
        st.subheader("PCA cumulative explained variance (numeric features)")
        st.plotly_chart(px.line(y=rep["pca_curve"], x=range(1, len(rep["pca_curve"]) + 1),
                                labels={"x": "components", "y": "variance"}), use_container_width=True)

with t2:
    st.subheader("Recommended preprocessing")
    for tip in dd.recommend(rep):
        st.markdown(f"- {tip}")
    st.subheader("Suggested algorithms")
    for a in dd.suggest_algorithms(rep):
        st.markdown(f"- {a}")

with t3:
    if st.button("Train & compare (raw vs. treated)"):
        with st.spinner("Training models..."):
            raw = df.replace(r"^\s*$", pd.NA, regex=True).dropna(subset=[target])
            for c in rep["wrong_dtypes"]:           # baseline still needs numeric parse
                raw[c] = pd.to_numeric(raw[c], errors="coerce")
            raw_res, _ = dd.train_evaluate(raw, target, use_smote=False)
            clean = dd.treat(df, target, rep, clip)
            tr_res, art = dd.train_evaluate(clean, target, use_smote=use_smote)
            st.session_state["art"] = art
        l, r = st.columns(2)
        l.subheader("Before (untreated)"); l.dataframe(raw_res)
        r.subheader("After (Dataset Doctor)"); r.dataframe(tr_res)
        st.success(f"Cleaned shape: {clean.shape} (original {df.shape})")
        st.download_button("Download cleaned CSV", clean.to_csv(index=False), "cleaned.csv")

with t4:
    art = st.session_state.get("art")
    if not art:
        st.info("Train models in the previous tab first.")
    else:
        tree = [m for m in ("XGBoost", "Random Forest") if m in art["models"]]
        pick = st.selectbox("Model", tree)
        sv, Xs = dd.shap_values(art["models" ][pick], art["X_test"])
        import shap
        fig = plt.figure()
        shap.summary_plot(sv, Xs, show=False, max_display=15)
        st.pyplot(fig)

2026-11-09 12:23:32.779 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-09 12:23:32.780 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-09 12:23:32.784 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-09 12:23:32.787 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-09 12:23:32.790 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-09 12:23:32.793 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-09 12:23:32.796 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-09 12:23:32.809 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [15]:
%%writefile app.py
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import streamlit as st
import os

import doctor as dd

st.set_page_config(page_title="AI Dataset Doctor", page_icon="ᾝ1‍⚕️", layout="wide")
st.title("ᾝ1‍⚕️ AI Dataset Doctor")
st.caption("Check whether your data is healthy and ML-ready before training anything.")

f = st.sidebar.file_uploader("Upload a CSV (e.g. Kaggle Telco Customer Churn)", type="csv")

# Fallback mechanism so the app has data even before uploading a file
if f is None:
    sample_path = "/content/sample_data/california_housing_test.csv"
    if os.path.exists(sample_path):
        st.info(f"Running in baseline mode. Loaded sample dataset: {sample_path}")
        df = pd.read_csv(sample_path)
    else:
        st.info("Upload a CSV to begin. Suggested: Kaggle → *blastchar/telco-customer-churn*.")
        st.stop()
else:
    df = pd.read_csv(f)

target = st.sidebar.selectbox("Target column", df.columns, index=len(df.columns) - 1)
use_smote = st.sidebar.checkbox("Apply SMOTE (train set only)", True)
clip = st.sidebar.checkbox("Clip outliers (IQR)", True)

rep = dd.diagnose(df, target)
t1, t2, t3, t4 = st.tabs(["ᾧA Health Report", "ὈA Prescription", "Ὄ8 Model Evaluation", "ὐD SHAP"])

with t1:
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Health score", f"{rep['health_score']}/100")
    c2.metric("Rows × Cols", f"{rep['rows']} × {rep['cols']}")
    c3.metric("Duplicates", rep["duplicates"])
    c4.metric("Imbalance ratio", f"{rep['imbalance_ratio']}:1")

    a, b = st.columns(2)
    with a:
        st.subheader("Missing values (%)")
        st.write(rep["missing"] or "None ✅")
        st.subheader("Wrong data types")
        st.write(rep["wrong_dtypes"] or "None ✅")
        st.subheader("Irrelevant features")
        st.write(rep["irrelevant"] or "None ✅")
    with b:
        st.subheader("Class distribution")
        cd = pd.Series(rep["class_dist"]).rename_axis("class").reset_index(name="%")
        st.plotly_chart(px.bar(cd, x="class", y="%", color="class"), use_container_width=True)

    st.subheader("Outliers (IQR / Z-score counts)")
    st.dataframe(pd.DataFrame(rep["outliers"]).T if rep["outliers"] else pd.DataFrame())
    st.subheader("Highly correlated pairs")
    st.write(rep["high_corr"] or "None ✅")
    if "pca_curve" in rep:
        st.subheader("PCA cumulative explained variance (numeric features)")
        st.plotly_chart(px.line(y=rep["pca_curve"], x=range(1, len(rep["pca_curve"]) + 1),
                                labels={"x": "components", "y": "variance"}), use_container_width=True)

with t2:
    st.subheader("Recommended preprocessing")
    for tip in dd.recommend(rep):
        st.markdown(f"- {tip}")
    st.subheader("Suggested algorithms")
    for a in dd.suggest_algorithms(rep):
        st.markdown(f"- {a}")

with t3:
    if st.button("Train & compare (raw vs. treated)"):
        with st.spinner("Training models..."):
            raw = df.replace(r"^\s*$", pd.NA, regex=True).dropna(subset=[target])
            for c in rep["wrong_dtypes"]:           # baseline still needs numeric parse
                raw[c] = pd.to_numeric(raw[c], errors="coerce")
            raw_res, _ = dd.train_evaluate(raw, target, use_smote=False)
            clean = dd.treat(df, target, rep, clip)
            tr_res, art = dd.train_evaluate(clean, target, use_smote=use_smote)
            st.session_state["art"] = art
        l, r = st.columns(2)
        l.subheader("Before (untreated)"); l.dataframe(raw_res)
        r.subheader("After (Dataset Doctor)"); r.dataframe(tr_res)
        st.success(f"Cleaned shape: {clean.shape} (original {df.shape})")
        st.download_button("Download cleaned CSV", clean.to_csv(index=False), "cleaned.csv")

with t4:
    art = st.session_state.get("art")
    if not art:
        st.info("Train models in the previous tab first.")
    else:
        tree = [m for m in ("XGBoost", "Random Forest") if m in art["models"]]
        pick = st.selectbox("Model", tree)
        sv, Xs = dd.shap_values(art["models"][pick], art["X_test"])
        import shap
        fig = plt.figure()
        shap.summary_plot(sv, Xs, show=False, max_display=15)
        st.pyplot(fig)

Writing app.py


In [ ]:
#@title Launch Streamlit Web App using Localtunnel
# We fetch your public IP because Localtunnel will request it as a security password.
import urllib.request
print("Your Tunnel Password (IP Address) is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

# Install localtunnel globally to route the port
!npm install -g localtunnel

# Run the streamlit application in the background and start localtunnel to expose port 8501
import subprocess
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
!lt --port 8501

Your Tunnel Password (IP Address) is: 34.186.7.12
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 6s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋your url is: https://big-buses-make.loca.lt
